
# Implementasi Recipe *AI Cookbook untuk Earth Sciences*

Notebook ini mengimplementasikan beberapa **recipe yang tercantum di buku** sebagai satu workflow Python yang saling terhubung.

Recipe yang digunakan:

1. **R-IO-001 — Membaca dan Merangkum Station CSV**
2. **R-ANA-001 — Exploratory Seasonal Summary**
3. **R-VIS-001 — Membuat Time-Series Figure yang Informatif**
4. **R-CAL-002 — Menghitung Anomaly terhadap Baseline**
5. **R-CAL-003 — Lag Correlation untuk Time Series**
6. **R-MOD-001 — Simulasi Difusi Panas Satu Dimensi**

Buku menggunakan alur **Problem → Minimal Theory → Prompt → Code → Result → Optional Check**. Notebook ini mengikuti pola tersebut, tetapi contoh datanya dibuat **sintetis** agar dapat dijalankan tanpa koneksi internet.

> **Catatan:** hasil notebook adalah hasil eksekusi data sintetis, bukan observasi lapangan.



## 1. Persiapan lingkungan

Library yang digunakan dibuat sederhana agar notebook mudah dijalankan di Jupyter Notebook, JupyterLab, maupun VS Code.

- `numpy` → komputasi numerik
- `pandas` → pengelolaan data tabular dan time series
- `matplotlib` → visualisasi
- `scipy.special` → solusi analitik difusi panas untuk verification


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import erfc
from pathlib import Path

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11

rng = np.random.default_rng(42)
print("Environment siap.")



# Recipe 1 — R-IO-001: Membaca dan Merangkum Station CSV

### Problem
Data pengamatan sering berada dalam CSV. Sebelum melakukan analisis, struktur file, jumlah data, missing value, dan ringkasan per stasiun perlu diperiksa.

Contoh berikut membuat **fixture sintetis** dengan kolom `station_id`, `timestamp`, dan `rainfall_mm`, lalu membaca kembali file tersebut.


In [ ]:

# Membuat fixture CSV sintetis
dates = pd.date_range("2026-01-01", periods=365, freq="D")
station_ids = ["STA01", "STA02", "STA03"]
frames = []

for station, phase in zip(station_ids, [0.0, 0.7, 1.3]):
    seasonal = 6 + 4 * np.sin(2 * np.pi * np.arange(365) / 365 + phase)
    wet_signal = 5 * np.maximum(
        0, np.sin(2 * np.pi * np.arange(365) / 90 + phase)
    )
    noise = rng.normal(0, 1.5, 365)
    rainfall = np.clip(seasonal + wet_signal + noise, 0, None)

    frames.append(pd.DataFrame({
        "station_id": station,
        "timestamp": dates,
        "rainfall_mm": rainfall
    }))

station_df = pd.concat(frames, ignore_index=True)

# Missing value untuk demonstrasi QA/QC
missing_idx = rng.choice(station_df.index, size=12, replace=False)
station_df.loc[missing_idx, "rainfall_mm"] = np.nan

csv_path = Path("synthetic_station_rainfall.csv")
station_df.to_csv(csv_path, index=False)

print(f"CSV dibuat: {csv_path}")
station_df.head()


In [ ]:

# Membaca kembali CSV
df = pd.read_csv(csv_path, parse_dates=["timestamp"])

print("Ukuran data :", df.shape)
print("Kolom       :", list(df.columns))

print("\nMissing value:")
print(df.isna().sum())

print("\nJumlah observasi per station:")
print(df.groupby("station_id").size())

summary_station = (
    df.groupby("station_id")["rainfall_mm"]
      .agg(["count", "mean", "std", "min", "max"])
      .round(2)
)

summary_station



### Visualisasi inspection

Plot ini digunakan untuk memeriksa pola umum data dan kemungkinan gap. Plot belum dipakai untuk membuat claim geofisika.


In [ ]:

fig, ax = plt.subplots()

for station, group in df.groupby("station_id"):
    ax.plot(
        group["timestamp"],
        group["rainfall_mm"],
        linewidth=1,
        label=station
    )

ax.set_title("R-IO-001 — Synthetic Station Rainfall")
ax.set_xlabel("Waktu")
ax.set_ylabel("Curah hujan (mm)")
ax.legend(title="Station")

fig.autofmt_xdate()
plt.show()



# Recipe 2 — R-ANA-001: Exploratory Seasonal Summary

### Problem
Setelah data dibaca, data dapat diringkas menurut waktu untuk melihat pola musiman. Implementasi berikut memakai rata-rata bulanan.

Karena dataset dibuat sintetis, pola yang terlihat hanya menggambarkan perilaku fixture, bukan klimatologi wilayah tertentu.


In [ ]:

df["month"] = df["timestamp"].dt.month

monthly = (
    df.groupby(["station_id", "month"], as_index=False)["rainfall_mm"]
      .mean()
      .rename(columns={"rainfall_mm": "mean_rainfall_mm"})
)

monthly.head(12)


In [ ]:

fig, ax = plt.subplots()

for station, group in monthly.groupby("station_id"):
    ax.plot(
        group["month"],
        group["mean_rainfall_mm"],
        marker="o",
        linewidth=2,
        label=station
    )

ax.set_title("R-ANA-001 — Rata-rata Curah Hujan Bulanan")
ax.set_xlabel("Bulan")
ax.set_ylabel("Rata-rata curah hujan (mm/hari)")
ax.set_xticks(range(1, 13))
ax.legend(title="Station")

plt.show()



### Optional check

Jumlah sampel valid per bulan ditampilkan agar pembaca dapat melihat apakah suatu rata-rata dibangun dari jumlah observasi yang sama.


In [ ]:

monthly_count = (
    df.groupby(["station_id", "month"])["rainfall_mm"]
      .count()
      .unstack(fill_value=0)
)

monthly_count



# Recipe 3 — R-VIS-001: Time-Series Figure yang Informatif

Recipe ini berfokus pada **figure hierarchy**, unit axis, dan gap data. Buku menyarankan agar missing interval tidak dihubungkan seolah-olah terdapat observasi di antaranya.

Pada contoh ini dibuat gap demonstrasi pada `STA01`.


In [ ]:

plot_df = df[df["station_id"] == "STA01"].copy()

# Gap demonstrasi selama 10 hari
gap_start = pd.Timestamp("2026-06-10")
gap_end = pd.Timestamp("2026-06-19")
gap_mask = plot_df["timestamp"].between(gap_start, gap_end)
plot_df.loc[gap_mask, "rainfall_mm"] = np.nan

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(
    plot_df["timestamp"],
    plot_df["rainfall_mm"],
    linewidth=1.5,
    label="STA01"
)

ax.set_title("R-VIS-001 — Time Series dengan Missing Interval")
ax.set_xlabel("Tanggal")
ax.set_ylabel("Curah hujan (mm)")
ax.legend()

fig.autofmt_xdate()
plt.show()



**Pembacaan figure:** garis terputus pada interval kosong berasal dari `NaN`, bukan interpolasi. Dengan demikian, visualisasi tidak menciptakan observasi yang sebenarnya tidak tersedia.



# Recipe 4 — R-CAL-002: Menghitung Anomaly terhadap Baseline

### Minimal theory

Buku mendefinisikan anomaly sederhana sebagai

\[
x'_t = x_t - \bar{x}_b
\]

dengan \(x_t\) sebagai observation dan \(\bar{x}_b\) sebagai rata-rata baseline.

Anomaly positif menunjukkan observation berada di atas baseline, sedangkan anomaly negatif menunjukkan observation berada di bawah baseline. Periode baseline harus dicatat karena perubahan baseline dapat mengubah nilai anomaly.


In [ ]:

# Baseline: rata-rata STA01 pada Januari–Maret
sta = df[df["station_id"] == "STA01"].copy()

baseline_period = sta[sta["timestamp"].dt.month.isin([1, 2, 3])]
baseline_mean = baseline_period["rainfall_mm"].mean()
baseline_n = baseline_period["rainfall_mm"].count()

sta["anomaly_mm"] = sta["rainfall_mm"] - baseline_mean

print(f"Baseline mean = {baseline_mean:.3f} mm")
print(f"Sample baseline valid = {baseline_n}")

sta[["timestamp", "rainfall_mm", "anomaly_mm"]].head()


In [ ]:

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(
    sta["timestamp"],
    sta["anomaly_mm"],
    linewidth=1.4,
    label="Anomaly"
)

ax.axhline(0, linestyle="--", linewidth=1)
ax.set_title("R-CAL-002 — Anomali terhadap Baseline Jan–Mar")
ax.set_xlabel("Tanggal")
ax.set_ylabel("Anomaly (mm)")
ax.legend()

fig.autofmt_xdate()
plt.show()



### Check anomaly

Pemeriksaan dilakukan pada dua hal: formula harus identik dengan \(x_t-\bar{x}_b\), dan nilai anomaly harus nol ketika observation sama dengan baseline.


In [ ]:

test_value = baseline_mean
test_anomaly = test_value - baseline_mean

assert np.isclose(test_anomaly, 0.0)
assert np.allclose(
    sta["anomaly_mm"].dropna(),
    (sta["rainfall_mm"] - baseline_mean).dropna()
)

print("Check R-CAL-002: PASS")



# Recipe 5 — R-CAL-003: Lag Correlation untuk Time Series

### Problem
Dua variabel dapat menunjukkan hubungan yang tertunda. Recipe ini menghitung korelasi pada beberapa lag menggunakan pasangan data yang valid.

### Konvensi lag

Notebook menetapkan **lag positif berarti `x` mendahului `y`**. Konvensi harus ditetapkan sebelum membaca puncak korelasi.

Lag correlation menunjukkan association pada offset waktu tertentu, bukan bukti sebab-akibat.


In [ ]:

# Dua time series sintetis dengan delay yang diketahui
n = 300
time = pd.date_range("2026-01-01", periods=n, freq="D")

x = np.sin(2 * np.pi * np.arange(n) / 45) + rng.normal(0, 0.15, n)

true_delay = 7
y = np.roll(x, true_delay) + rng.normal(0, 0.20, n)
y[:true_delay] = np.nan

lag_df = pd.DataFrame({
    "time": time,
    "x": x,
    "y": y
})

lag_df.head()


In [ ]:

def lag_correlation(x, y, lags):
    # Pearson correlation untuk pasangan valid pada setiap lag.
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    records = []

    for lag in lags:
        if lag > 0:
            x_lag = x[:-lag]
            y_lag = y[lag:]
        elif lag < 0:
            x_lag = x[-lag:]
            y_lag = y[:lag]
        else:
            x_lag = x
            y_lag = y

        valid = np.isfinite(x_lag) & np.isfinite(y_lag)

        if valid.sum() < 3:
            corr = np.nan
        else:
            corr = np.corrcoef(x_lag[valid], y_lag[valid])[0, 1]

        records.append({
            "lag_days": lag,
            "correlation": corr,
            "valid_pairs": int(valid.sum())
        })

    return pd.DataFrame(records)

lags = np.arange(-30, 31)
lag_result = lag_correlation(lag_df["x"], lag_df["y"], lags)

lag_result.head()


In [ ]:

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    lag_result["lag_days"],
    lag_result["correlation"],
    marker="o",
    markersize=3
)

best = lag_result.loc[lag_result["correlation"].idxmax()]

ax.axvline(
    best["lag_days"],
    linestyle="--",
    linewidth=1,
    label=f"Maximum = {best['lag_days']:.0f} hari"
)

ax.set_title("R-CAL-003 — Lag Correlation")
ax.set_xlabel("Lag (hari)")
ax.set_ylabel("Pearson correlation")
ax.legend()

plt.show()

print("Lag dengan korelasi maksimum:")
print(best)



### Interpretasi terbatas

Puncak korelasi pada lag tertentu menunjukkan bahwa kedua sinyal paling mirip ketika dibandingkan dengan offset tersebut. Pada penelitian nyata, interpretasi harus mempertimbangkan autocorrelation, filtering, sampling interval, jumlah lag, dan minimum valid pair.



# Recipe 6 — R-MOD-001: Simulasi Difusi Panas Satu Dimensi

### Minimal theory

Model difusi panas yang digunakan:

\[
rac{\partial T}{\partial t}
=
\kappa
rac{\partial^2 T}{\partial z^2}
\]

dengan \(T\) dalam °C, \(z\) sebagai depth positif ke bawah, \(t\) dalam detik, dan \(\kappa\) dalam m²/s.

Untuk explicit finite difference digunakan kriteria stabilitas

\[
r = rac{\kappa\Delta t}{\Delta z^2} \leq rac{1}{2}.
\]

Sebelum solver dijalankan, notebook menghitung `r` menggunakan spacing aktual dan time step aktual.


In [ ]:

# Parameter model
depth = 5.0
dz_requested = 0.10
duration_days = 30
dt_requested = 12 * 3600
kappa = 0.6e-6

T_initial = 22.0
T_surface = 30.0
T_bottom = 22.0

# Grid aktual
z = np.arange(0, depth + dz_requested, dz_requested)
dz = z[1] - z[0]

duration = duration_days * 24 * 3600

# dt dipilih agar stabil dan final time tepat
max_dt = 0.5 * dz**2 / kappa
dt_limit = min(dt_requested, max_dt)
n_steps = int(np.ceil(duration / dt_limit))
dt = duration / n_steps

r = kappa * dt / dz**2

print(f"Grid points       : {len(z)}")
print(f"dz aktual         : {dz:.4f} m")
print(f"dt aktual         : {dt/3600:.3f} jam")
print(f"Stability ratio r : {r:.4f}")

assert r <= 0.5 + 1e-12


In [ ]:

# Explicit finite-difference solver
n_save = 121
save_every = max(1, n_steps // (n_save - 1))

T = np.full(len(z), T_initial, dtype=float)
T[0] = T_surface
T[-1] = T_bottom

saved_T = [T.copy()]
saved_t = [0.0]

for step in range(1, n_steps + 1):
    T_new = T.copy()

    T_new[1:-1] = (
        T[1:-1]
        + r * (T[2:] - 2*T[1:-1] + T[:-2])
    )

    T_new[0] = T_surface
    T_new[-1] = T_bottom

    T = T_new

    if step % save_every == 0 or step == n_steps:
        saved_T.append(T.copy())
        saved_t.append(step * dt)

temperature = np.array(saved_T)
time_seconds = np.array(saved_t)
time_days = time_seconds / 86400

print("Simulation selesai.")
print("Shape temperature:", temperature.shape)


In [ ]:

# Heatmap
fig, ax = plt.subplots(figsize=(10, 6))

im = ax.imshow(
    temperature.T,
    aspect="auto",
    origin="upper",
    extent=[time_days.min(), time_days.max(), z.max(), z.min()]
)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Temperature (°C)")

ax.set_title("R-MOD-001 — One-Dimensional Heat Diffusion")
ax.set_xlabel("Waktu (hari)")
ax.set_ylabel("Depth (m, positif ke bawah)")

plt.show()


In [ ]:

# Profil temperatur pada beberapa waktu
selected_days = [0, 5, 15, 30]

fig, ax = plt.subplots(figsize=(8, 6))

for day in selected_days:
    idx = np.abs(time_days - day).argmin()
    ax.plot(
        temperature[idx],
        z,
        linewidth=2,
        label=f"{time_days[idx]:.0f} hari"
    )

ax.invert_yaxis()
ax.set_title("Profil Temperatur pada Beberapa Waktu")
ax.set_xlabel("Temperature (°C)")
ax.set_ylabel("Depth (m)")
ax.legend()

plt.show()



## Verification: numerical profile vs analytic solution

Sebagai check tambahan, profil akhir dibandingkan dengan bentuk solusi semi-infinite untuk *step surface warming*:

\[
T(z,t)=T_i+(T_s-T_i)
\operatorname{erfc}
\left(
rac{z}{2\sqrt{\kappa t}}
ight).
\]

Node batas bawah dikeluarkan dari RMSE karena model numerik mempunyai **fixed bottom boundary**, sedangkan solusi semi-infinite tidak mempunyai batas bawah.


In [ ]:

final_T = temperature[-1]
t_final = time_seconds[-1]

analytic = (
    T_initial
    + (T_surface - T_initial)
    * erfc(z / (2 * np.sqrt(kappa * t_final)))
)

comparison = np.arange(len(z) - 1)

rmse = np.sqrt(
    np.mean(
        (final_T[comparison] - analytic[comparison])**2
    )
)

max_abs_error = np.max(
    np.abs(final_T[comparison] - analytic[comparison])
)

print(f"Final time           : {t_final/86400:.2f} hari")
print(f"RMSE                 : {rmse:.4f} °C")
print(f"Maximum absolute err : {max_abs_error:.4f} °C")


In [ ]:

fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(
    final_T,
    z,
    linewidth=2,
    label="Numerical"
)

ax.plot(
    analytic,
    z,
    linestyle="--",
    linewidth=2,
    label="Semi-infinite analytic"
)

ax.invert_yaxis()
ax.set_title("R-MOD-001 — Numerical vs Analytic Profile")
ax.set_xlabel("Temperature (°C)")
ax.set_ylabel("Depth (m)")
ax.legend()

plt.show()



# Ringkasan workflow

| Recipe | Input | Keluaran |
|---|---|---|
| R-IO-001 | Station CSV | Struktur dan summary data |
| R-ANA-001 | Time series | Ringkasan bulanan |
| R-VIS-001 | Time series | Figure dengan gap |
| R-CAL-002 | Observation + baseline | Anomaly series |
| R-CAL-003 | Dua time series | Lag-correlation table dan plot |
| R-MOD-001 | Parameter thermal | Simulasi temperatur, heatmap, dan analytic check |

### Batas interpretasi

- Data pada notebook adalah **sintetis**, bukan data lapangan.
- Anomaly bergantung pada baseline yang dipilih.
- Lag correlation mengukur association pada offset waktu, bukan causality.
- Stability check dan analytic comparison pada model panas adalah **verification**, bukan validasi terhadap data lapangan.
- Untuk hasil penelitian, simpan input, unit, parameter, configuration, code, output, dan catatan interpretasi agar workflow dapat direproduksi.



## Sumber utama

**Assagaf, A. (2026). *AI Cookbook untuk Earth Sciences: Prompt, Python, Scientific Modeling, dan Reproducible Workflows*. Edisi Pertama.**

Recipe yang diimplementasikan di notebook ini tercantum dalam katalog recipe buku dan dijelaskan pada bagian Scientific Calculations, Scientific Visualization, Local Data/Scientific I/O, dan Developing Scientific Models.
